In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
data = [
    ("Alice", 25, "New York"),
    ("Bob", 30, "San Francisco"),
    ("Charlie", 35, "Chicago")
]

schema = StructType([                                               # StructType represents the entire schema of the DataFrame.
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),                        # Each StructField defines one column.
    StructField("city", StringType(), True)
])

df_narrow = spark.createDataFrame(data, schema=schema)
df_wide = spark.createDataFrame(data, schema=schema)

## NARROW TRANSFORMATION

In [0]:
df_narrow = df_narrow.filter(col('city') == "New York")

In [0]:
df_narrow.show()

+-----+---+--------+
| name|age|    city|
+-----+---+--------+
|Alice| 25|New York|
+-----+---+--------+



## WIDE TRANSFORMATION

In [0]:
df_wide = df_wide.groupBy('city').agg(max(col('age')))      # groupBy() requires Spark to move/shuffle data between partitions

In [0]:
df_wide.show()

+-------------+--------+
|         city|max(age)|
+-------------+--------+
|     New York|      25|
|San Francisco|      30|
|      Chicago|      35|
+-------------+--------+



In [0]:
df_wide.explain('extended')

== Parsed Logical Plan ==
'Aggregate ['city], ['city, unresolvedalias('max('age))]
+- LocalRelation [name#11250, age#11251, city#11252]

== Analyzed Logical Plan ==
city: string, max(age): int
Aggregate [city#11252], [city#11252, max(age#11251) AS max(age)#11297]
+- LocalRelation [name#11250, age#11251, city#11252]

== Optimized Logical Plan ==
Aggregate [city#11252], [city#11252, max(age#11251) AS max(age)#11297]
+- LocalRelation [age#11251, city#11252]

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(limit=None, keys=[city#11252], functions=[max(age#11251)], output=[city#11252, max(age)#11297])
         +- PhotonRowToColumnar
            +- LocalTableScan [age#11251, city#11252]

== Photon Explanation ==
The query is fully supported by Photon.
